[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/04_retrieval/04_retrieval.ipynb)

# 04 · 检索入上下文（agent 的 RAG）

目标：用**纯 Python 标准库**从零写出一个检索器——**toy embedding → 余弦相似 → top-k → 重排 → 按注入预算拼进上下文**，全程 `assert` 验证、与暴力扫描对拍，**无 numpy、无需 API key**。

路线：toy embedding → 余弦(守除零) → top-k(对拍暴力解) → 注入预算 → 重排(两段式) → ✏️ 练习 → 📖 答案 → 🧪 真实语料胶囊。

> 心智模型：**检索 = 在海量外部知识里，按相关性挑出当前最该看的几页，塞进有限的窗口预算**。难点在『怎么量化相关性』与『top-k 与预算之间怎么取舍』。

## 1 · toy embedding：把文本变成词频向量

真实系统用学习到的 embedding 模型（DPR 等）。本课用一个**确定性词频向量**：把每个词哈希到固定维度的某一格并计数。它只懂词面、不懂同义词，但**完全确定、可断言**，足以讲清检索的控制流。

> 真实里换成 `embedding 模型 / API`，但『同输入同向量』这一性质 toy 与真实一致。

In [ ]:
import re, hashlib, math, json

def embed(text, dim=32):
    '''确定性 toy embedding: 词 -> 哈希到某一格 -> 词频计数。纯标准库。'''
    vec = [0.0] * dim
    for tok in re.findall(r'[\w]+', text.lower()):
        idx = int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim
        vec[idx] += 1.0
    return vec

v1 = embed('the cat sat on the mat')
v2 = embed('the cat sat on the mat')
v3 = embed('')                       # 空文本 -> 全零向量
print('维度:', len(v1), '| 非零格数:', sum(1 for x in v1 if x))
print('同输入同向量?', v1 == v2)
print('空文本向量全零?', all(x == 0.0 for x in v3))
assert len(v1) == 32 and v1 == v2            # 确定性、定维
assert sum(v1) == 6.0                          # 6 个词(含两个 the) -> 词频和=6
assert all(x == 0.0 for x in v3)              # 空文本=零向量(下一节要守除零)
print('✅ toy embedding: 确定性、定维、词频和正确，空文本=零向量')

## 2 · 余弦相似度：从零实现，守住除零

$\cos(a,b)=\dfrac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}$：点积除以两个模长之积，范围 $[-1,1]$（词频非负时落在 $[0,1]$）。

**最致命的 bug 是除零**：任一向量全零(模长 0)，分母为 0 会崩。必须显式判断 -> 返回 0（无法比较视作不相关）。

In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def norm(a):
    return math.sqrt(sum(x * x for x in a))

def cosine(a, b):
    '''余弦相似度, 守住除零: 任一向量模长为 0 -> 返回 0.0。'''
    na, nb = norm(a), norm(b)
    if na == 0.0 or nb == 0.0:          # 关键: 零向量直接判 0, 不崩
        return 0.0
    return dot(a, b) / (na * nb)

qa = embed('weather forecast beijing')
doc_same = embed('beijing weather forecast today')   # 高度重合
doc_diff = embed('quarterly financial report')        # 完全不同主题
zero = embed('')
s_same, s_diff, s_zero = cosine(qa, doc_same), cosine(qa, doc_diff), cosine(qa, zero)
print(f'与同主题文档余弦 = {s_same:.3f}')
print(f'与异主题文档余弦 = {s_diff:.3f}')
print(f'与零向量余弦     = {s_zero:.3f} (守住除零, 不崩)')
assert abs(cosine(qa, qa) - 1.0) < 1e-9       # 自己和自己 = 1
assert s_same > s_diff                          # 同主题更相关
assert s_zero == 0.0                            # 除零被守住
assert -1.0 - 1e-9 <= s_diff <= 1.0 + 1e-9    # 落在合法范围
print('✅ 余弦: 自相似=1、同主题>异主题、零向量返回0(不崩)')

## 3 · top-k 检索：对每个候选算余弦、排序取前 k（对拍暴力解）

建一个小语料，对查询算与每个文档的余弦，按相似度降序取 top-k。

**正确性裁判 = 暴力扫描**：构造一个『谁该排第一』显而易见的查询，断言 top-k 与『全算一遍再排』一致。排序用**确定的 tie-breaker（原始下标）**，保证可复现。

In [ ]:
CORPUS = [
    'beijing weather forecast sunny today',          # 0 天气
    'shanghai weather rainy tomorrow forecast',      # 1 天气
    'quarterly financial report revenue growth',     # 2 财务
    'python list comprehension tutorial code',       # 3 编程
    'machine learning model training dataset',       # 4 ML
    'beijing travel guide great wall tourist',       # 5 旅游(含beijing)
]
DOC_VECS = [embed(doc) for doc in CORPUS]            # 预先嵌入(真实里存进向量库)

def retrieve(query, doc_vecs, k):
    '''线性扫描: 对每个候选算余弦, 降序取 top-k。
       返回 [(idx, score), ...]; tie-breaker=原始下标(确定可复现)。'''
    qv = embed(query)
    scored = [(i, cosine(qv, dv)) for i, dv in enumerate(doc_vecs)]
    # 降序按 score, 同分按下标升序 -> 用 (-score, idx) 作排序键
    scored.sort(key=lambda t: (-t[1], t[0]))
    return scored[:k]

def brute_force_topk(query, corpus, k):
    '''Oracle: 最朴素地全算一遍再排序, 当裁判对拍。'''
    qv = embed(query)
    pairs = []
    for i, doc in enumerate(corpus):
        pairs.append((i, cosine(qv, embed(doc))))
    pairs.sort(key=lambda t: (-t[1], t[0]))
    return pairs[:k]

q = 'beijing weather'
top = retrieve(q, DOC_VECS, k=3)
print('查询:', q)
for i, s in top:
    print(f'  doc[{i}] score={s:.3f}  {CORPUS[i]}')
# doc0(beijing+weather 两词都中)余弦最高, 必排第一 —— 数学保证, 不受哈希碰撞影响
assert top[0][0] == 0, '与查询共享最多词的应排第一'
# 与暴力 oracle 完全一致(下标序列) —— 这是检索器的对拍裁判
assert [i for i, _ in top] == [i for i, _ in brute_force_topk(q, CORPUS, 3)]
# 分数严格按降序(top-k 的定义)
assert all(top[j][1] >= top[j+1][1] for j in range(len(top)-1))
# 与查询毫无共同词、且无哈希碰撞的文档余弦为 0(我们显式核对其得分确为 0)
all_scores = dict((i, cosine(embed(q), dv)) for i, dv in enumerate(DOC_VECS))
zero_docs = [i for i, sc in all_scores.items() if sc == 0.0]
print('与查询余弦为0(完全不相关)的文档:', zero_docs)
assert all(i not in [j for j, _ in top] for i in zero_docs), '余弦为0的文档不该进 top-k'
print('✅ top-k: 正确文档排第一、与暴力扫描一致、零相关文档被排除')

## 4 · 注入预算：top-k 之上的硬约束

检索回来的片段要拼进上下文，而上下文有预算。**召回了 ≠ 注入得了**：按相关性从高到低逐个累加 token，装不下的丢弃。真正进上下文的是『top-k 里、且累计 token 不超预算的前几个』。

In [ ]:
def count_tokens(text):
    '''与全课一致的确定性近似计数(真实换成 messages.count_tokens)。'''
    return sum(max(1, (len(w) + 3) // 4) for w in text.split()) if text else 0

def select_within_budget(ranked_idxs, corpus, budget):
    '''贪心装包: 候选已按相关性排序, 从最相关开始累加 token,
       不超 budget 就纳入, 超了就停。返回 (选中的下标列表, 用掉的token)。'''
    chosen, used = [], 0
    for i in ranked_idxs:
        t = count_tokens(corpus[i])
        if used + t > budget:
            break                       # 超预算即停(最简策略)
        chosen.append(i)
        used += t
    return chosen, used

def retrieve_and_inject(query, doc_vecs, corpus, k, budget):
    '''完整检索->注入: top-k 召回, 再按注入预算筛选。'''
    top = retrieve(query, doc_vecs, k)
    ranked = [i for i, _ in top]
    chosen, used = select_within_budget(ranked, corpus, budget)
    return chosen, used

q = 'beijing weather'
print('每条文档的 token 数:', [count_tokens(d) for d in CORPUS])  # 各约 10-12
# 预算 22 ~ 只够最相关的前 2 条(每条约10), 第3条加进来就超 -> 演示『召回了≠注入得了』
chosen, used = retrieve_and_inject(q, DOC_VECS, CORPUS, k=4, budget=22)
print('top-4 召回, 注入预算=22 ->实际注入:', chosen, '用掉', used, 'tokens')
assert used <= 22, '注入总 token 必须 <= 预算(硬约束)'
assert chosen[0] == 0, '最相关的一定先被纳入'
assert len(chosen) < 4, '预算不够装下全部 top-4 -> 应只注入前几个(召回了≠注入得了)'
# 预算给足时, top-k 应全部注入
chosen_big, _ = retrieve_and_inject(q, DOC_VECS, CORPUS, k=3, budget=1000)
assert len(chosen_big) == 3
# 预算为 0 -> 一个都注入不了
assert retrieve_and_inject(q, DOC_VECS, CORPUS, k=4, budget=0) == ([], 0)
print('✅ 注入预算: 总token<=预算、最相关优先、预算不足只注入前几个')

## 5 · 重排：两段式召回，把噪声排到后面

纯余弦快但糙：它靠词频，会把**靠重复查询词灌水**的文档排得很前——哪怕那段又长又啰嗦。**两段式**：先用便宜余弦召回 top-N(放宽)，再用更精的打分重排出 top-k(收紧)。这里 toy reranker 用『**覆盖了几个不同的查询词**(coverage) × **越短越好**(brevity)』当更贴近相关性的信号——它奖励『精准覆盖』、惩罚『灌水重复』。

下面构造一个『纯余弦把灌水文档排第一、重排能把精准文档顶上来』的例子。

In [ ]:
def rerank_score(query, doc):
    '''toy reranker: 比纯余弦更贴近相关性。
       coverage = 覆盖了几个『不同的』查询词 / 查询词总数  (重复不加分 -> 抗灌水)
       brevity  = 1/(1+0.05*文档词数)                      (越短越聚焦 -> 抗噪声)
       两者相乘: 既要覆盖查询、又要简洁精准。'''
    q_terms = set(re.findall(r'[\w]+', query.lower()))
    d_terms = re.findall(r'[\w]+', doc.lower())
    coverage = len(q_terms & set(d_terms)) / len(q_terms)   # 不同查询词的覆盖率
    brevity = 1.0 / (1 + 0.05 * len(d_terms))               # 长度惩罚
    return coverage * brevity

def two_stage_retrieve(query, doc_vecs, corpus, n_recall, k):
    '''阶段1: 余弦召回 top-N(放宽); 阶段2: rerank 精排出 top-k(收紧)。'''
    recalled = [i for i, _ in retrieve(query, doc_vecs, n_recall)]   # 召回
    reranked = sorted(recalled, key=lambda i: (-rerank_score(query, corpus[i]), i))
    return reranked[:k]

# 一个 FAQ 风格小库; 查询='refund policy window'
CORPUS2 = [
    'refund policy window refund policy window refund policy window extra padding noise',  # 0 灌水: 含全部查询词但重复+噪声
    'refund policy window is 30 days',                                                     # 1 精准简洁, 覆盖全3词
    'shipping policy details here',                                                        # 2 只覆盖1词(policy)
    'account password reset guide',                                                        # 3 覆盖0词
    'refund window for returns explained',                                                 # 4 覆盖2词(refund,window)
]
VECS2 = [embed(d) for d in CORPUS2]
q = 'refund policy window'
cosine_only = [i for i, _ in retrieve(q, VECS2, k=3)]
two_stage = two_stage_retrieve(q, VECS2, CORPUS2, n_recall=5, k=3)
print('纯余弦 top-3 :', cosine_only, '(灌水文档 doc0 因词频高被排第一)')
print('两段式 top-3:', two_stage, '(重排把精准文档 doc1 顶到第一)')
# 纯余弦把灌水的 doc0 排第一; 重排把精准的 doc1 顶上来 -> 两个顺序确实不同
assert cosine_only[0] == 0, '纯余弦被灌水文档(词频高)带偏, 排第一'
assert two_stage[0] == 1, '重排把精准简洁的文档顶到第一'
assert cosine_only != two_stage, '重排确实改变了顺序(否则白做)'
# rerank 给精准文档的分 > 给灌水文档的分(覆盖率同为1, 但 doc1 更短)
assert rerank_score(q, CORPUS2[1]) > rerank_score(q, CORPUS2[0])
print('✅ 重排: 两段式(召回放宽+重排收紧), 把精准文档顶到灌水文档之前')

---
## ✏️ 练习 1：余弦相似度 + 守除零

**不看上面的实现**，自己写 `my_cosine(a, b)`：点积除以两个 L2 模长之积；**任一向量模长为 0 时返回 0.0**（不能崩、不能返回 nan）。

In [ ]:
def my_cosine(a, b):
    # TODO: 
    #   1. 算点积 dot = sum(x*y)
    #   2. 算两个模长 na, nb = sqrt(sum(x*x))
    #   3. 若 na==0 或 nb==0 -> return 0.0 (守除零!)
    #   4. 否则 return dot/(na*nb)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(my_cosine([1, 0, 1], [1, 0, 1]) - 1.0) < 1e-9    # 自相似=1
assert abs(my_cosine([1, 0], [0, 1]) - 0.0) < 1e-9          # 正交=0
assert my_cosine([0, 0, 0], [1, 2, 3]) == 0.0               # 零向量守住!
assert my_cosine([1, 1], [2, 2]) > 0.99                      # 同方向不同长度~1
import math as _m
assert not _m.isnan(my_cosine([0,0],[0,0]))                  # 两个零向量也不能 nan
print('✅ 练习 1 通过：余弦正确且零向量不崩、不 nan')

## ✏️ 练习 2：top-k 检索（与暴力解一致）

实现 `my_topk(query, doc_vecs, k)`：对每个候选算余弦，**降序**返回 `[(idx, score), ...]` 的前 k 个。同分时按**原始下标升序**（确定的 tie-breaker，保证可复现）。可复用上面的 `embed` / `cosine`。

In [ ]:
def my_topk(query, doc_vecs, k):
    # TODO:
    #   1. qv = embed(query)
    #   2. 对每个 (i, dv) 算 cosine(qv, dv) -> [(i, score), ...]
    #   3. 按 (-score, i) 排序(降序分数、同分按下标升序)
    #   4. 取前 k 个返回
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
res = my_topk('beijing weather', DOC_VECS, 3)
assert isinstance(res, list) and len(res) == 3
assert res[0][0] == 0                                # doc0 排第一
# 分数单调不增(已降序)
assert all(res[i][1] >= res[i+1][1] for i in range(len(res)-1))
# 与本课 retrieve 的下标序列一致
assert [i for i, _ in res] == [i for i, _ in retrieve('beijing weather', DOC_VECS, 3)]
print('✅ 练习 2 通过：top-k 降序、tie-break 确定、与参考一致')

## ✏️ 练习 3：注入预算（贪心装包）

实现 `inject_within_budget(ranked_idxs, corpus, budget)`：候选已按相关性排序，从最相关开始逐个累加 `count_tokens(corpus[i])`，**不超 budget 就纳入、超了就停**，返回 `(选中下标列表, 用掉的token)`。

In [ ]:
def inject_within_budget(ranked_idxs, corpus, budget):
    # TODO:
    #   chosen, used = [], 0
    #   依次遍历 ranked_idxs: t = count_tokens(corpus[i])
    #     若 used + t > budget -> break
    #     否则 chosen.append(i); used += t
    #   返回 (chosen, used)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ranked = [0, 1, 5, 2]   # 假设这是按相关性排好的下标
chosen, used = inject_within_budget(ranked, CORPUS, budget=14)
assert used <= 14                                    # 硬约束: 不超预算
assert chosen[0] == 0                                # 最相关先纳入
assert chosen == ranked[:len(chosen)]                # 按序前缀(贪心)
# 预算 0 -> 一个都装不下
assert inject_within_budget(ranked, CORPUS, budget=0) == ([], 0)
# 预算充足 -> 全装下
big_chosen, _ = inject_within_budget(ranked, CORPUS, budget=10000)
assert big_chosen == ranked
print('✅ 练习 3 通过：贪心装包、不超预算、按相关性前缀纳入')

## ✏️ 练习 4：两段式重排

实现 `my_two_stage(query, doc_vecs, corpus, n_recall, k)`：阶段1 用余弦 `retrieve` 召回 top-N 的下标；阶段2 用 `rerank_score` 对召回的下标**降序重排**（同分按下标升序），返回前 k 个下标。

In [ ]:
def my_two_stage(query, doc_vecs, corpus, n_recall, k):
    # TODO:
    #   1. recalled = retrieve(query, doc_vecs, n_recall) 的下标列表
    #   2. 按 (-rerank_score(query, corpus[i]), i) 对 recalled 排序
    #   3. 取前 k 个下标返回
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
q = 'refund policy window'
out = my_two_stage(q, VECS2, CORPUS2, n_recall=5, k=3)
assert isinstance(out, list) and len(out) == 3
assert out[0] == 1                                   # 精准简洁 doc1 -> 重排第一
# 重排确实改变了纯余弦的顺序(灌水 doc0 被压下去)
assert out != [i for i, _ in retrieve(q, VECS2, 3)]
# 只在召回池内重排(不会凭空冒出召回池外的)
recall_pool = [i for i, _ in retrieve(q, VECS2, 5)]
assert all(i in recall_pool for i in out)
# 重排后分数不增
assert all(rerank_score(q, CORPUS2[out[i]]) >= rerank_score(q, CORPUS2[out[i+1]])
           for i in range(len(out)-1))
print('✅ 练习 4 通过：两段式、重排改变顺序、只在召回池内重排、按精排分数降序')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_cosine(a, b):
    d = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0.0 or nb == 0.0:
        return 0.0
    return d / (na * nb)

In [ ]:
# 练习 2 参考答案
def my_topk(query, doc_vecs, k):
    qv = embed(query)
    scored = [(i, cosine(qv, dv)) for i, dv in enumerate(doc_vecs)]
    scored.sort(key=lambda t: (-t[1], t[0]))
    return scored[:k]

In [ ]:
# 练习 3 参考答案
def inject_within_budget(ranked_idxs, corpus, budget):
    chosen, used = [], 0
    for i in ranked_idxs:
        t = count_tokens(corpus[i])
        if used + t > budget:
            break
        chosen.append(i)
        used += t
    return chosen, used

In [ ]:
# 练习 4 参考答案
def my_two_stage(query, doc_vecs, corpus, n_recall, k):
    recalled = [i for i, _ in retrieve(query, doc_vecs, n_recall)]
    recalled.sort(key=lambda i: (-rerank_score(query, corpus[i]), i))
    return recalled[:k]

---
## 🧪 真实数据胶囊：一小段真实风格语料的端到端检索

下面用一段**贴近真实**的小知识库（FAQ 风格的条目，像你会喂给 RAG 的产品文档片段）跑完整链路：embed → 检索 → 重排 → 按预算注入 → 拼成一次 Messages API 请求的形状。

> 形状对照：真实里 `DOC_VECS` 存进**向量库(FAISS/ANN)**、`embed` 换成 **embedding 模型**，检索结果按预算拼进 **`system` 或一条 `user` 消息**——控制流与这里逐行对应。

In [ ]:
KB = [
    'refund policy: refunds are available within 30 days of purchase',     # 0
    'shipping: standard shipping takes 5 to 7 business days',              # 1
    'refund process: contact support with your order id to start a refund',# 2
    'account: reset your password from the login page',                    # 3
    'shipping cost: free shipping on orders over 50 dollars',              # 4
]
KB_VECS = [embed(d) for d in KB]

def build_rag_messages(query, kb, kb_vecs, k, budget, system='You are a support bot.'):
    '''完整 RAG 组装: 检索->按预算注入->拼成 system+messages(贴近 Messages API)。'''
    top = retrieve(query, kb_vecs, k)
    ranked = [i for i, _ in top]
    chosen, used = select_within_budget(ranked, kb, budget)
    context = '\n'.join(f'- {kb[i]}' for i in chosen)
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': f'参考资料:\n{context}\n\n问题: {query}'},
    ]
    return messages, chosen, used

q = 'how do i get a refund'
msgs, chosen, used = build_rag_messages(q, KB, KB_VECS, k=3, budget=40)
print('注入的条目下标:', chosen, '| 用掉 token:', used)
print('拼好的 user 消息:\n', msgs[1]['content'])
# refund 相关条目(0,2)应被检索到并注入
assert 0 in chosen and 2 in chosen, 'refund 相关条目应被召回'
assert 1 not in chosen and 3 not in chosen, '无关条目不该注入'
assert used <= 40 and msgs[0]['role'] == 'system'
print('✅ 胶囊: 真实风格 RAG 端到端 —— 检索->重排->按预算注入->拼成 Messages 形状')

**🧪 胶囊练习**：实现 `precision_at_k(retrieved_idxs, relevant_set)`：给定检索回的下标列表与『真正相关』的下标集合，返回 precision@k = `命中数 / len(retrieved_idxs)`（检索回的里有多少真相关）。这是评测检索质量的基本指标。

In [ ]:
def precision_at_k(retrieved_idxs, relevant_set):
    # TODO: 返回 (retrieved 里属于 relevant_set 的个数) / len(retrieved_idxs)
    #       len(retrieved_idxs)==0 时返回 0.0
    raise NotImplementedError

In [ ]:
# 自测
assert precision_at_k([0, 2, 1], {0, 2}) == 2 / 3      # 3个里2个相关
assert precision_at_k([0, 2], {0, 2}) == 1.0           # 全相关
assert precision_at_k([], {0, 2}) == 0.0               # 空->0
print('precision@3 of [0,2,1] =', round(precision_at_k([0,2,1], {0,2}), 3))
print('✅ 胶囊练习通过：precision@k 正确')

In [ ]:
# 📖 胶囊参考答案
def precision_at_k(retrieved_idxs, relevant_set):
    if not retrieved_idxs:
        return 0.0
    hits = sum(1 for i in retrieved_idxs if i in relevant_set)
    return hits / len(retrieved_idxs)

---
## 🔧 旁注：对应的真实检索栈长什么样（无 key 自动回退）

本课用 toy 词频 embedding + 线性扫描跑通的检索器，换成真实栈只是替换两个零件——**控制流一行不改**：

```python
# 真实里(伪代码, 需 key/依赖): embed 换成 embedding 模型, 线性扫描换成向量库
# import anthropic, faiss ...
# def embed(text): return embedding_model.encode(text)        # 学习到的向量
# index = faiss.IndexFlatIP(dim); index.add(doc_vecs)         # 向量库
# scores, idxs = index.search(embed(query), n_recall)         # 近似最近邻召回
# ... 然后 rerank / select_within_budget 与本课逐行相同 ...
#
# 把检索结果拼进上下文, 发给真实 Claude:
# resp = client.messages.create(
#     model='claude-sonnet-4-6', max_tokens=1024,
#     system='You are a support bot.',
#     messages=[{'role':'user','content': f'参考资料:\n{context}\n\n问题: {q}'}])
```

对应关系：toy `embed` ↔ embedding 模型、线性扫描 ↔ 向量库(FAISS/ANN)、`build_rag_messages` ↔ 真实 `messages` 组装。我们的 **余弦 / top-k / 重排 / 注入预算 逻辑原样适用**。

> 沿用模块 00 的 `get_llm()`：**有 `ANTHROPIC_API_KEY` 就调真实 Claude，没有就回退 MockLLM**——检索这一侧（embed/检索/注入）本就是纯本地计算，无论有没有 key 都照跑；只有最后『把注入好的上下文发给模型』那一步才用到 LLM，而它也无 key 自动回退、绝不阻断。

### 小结
- 检索 = 把窗口外的海量知识，按相关性、按预算精准注入窗口内（agent 的 RAG）。
- **embedding**：文本->向量，语义近->向量近；toy=词频向量(确定可断言)，真实=学习到的 embedding。
- **余弦相似**：点积/模长积，只看方向不看长度；**必须守住除零**(零向量返回0)。
- **top-k**：算余弦排序取前 k；k 是召回与噪声/预算的权衡；用暴力扫描对拍。
- **注入预算**：top-k 之上的硬约束——召回了≠注入得了，按相关性贪心装包、总 token 有界。
- **重排**：两段式(便宜召回放宽 + 昂贵精排收紧)，把噪声排到后面、提相关性。

下一站：**模块 05 · Prompt 缓存与长会话** —— 当这套机制让 agent 能长时间运行时，成本浮现为新约束，用缓存把稳定前缀的开销压下去。